# Jigsaw Toxic Comment Classification

## BiLSTM Model

### Objective
Train a Bidirectional LSTM model to classify toxic comments across six categories.

### Tasks
- Mount Google Drive and load tokenized artifacts
- Build GloVe embedding matrix
- Define and compile the BiLSTM architecture
- Train the model with early stopping
- Evaluate performance and generate predictions
- Save submission file to Google Drive

## Import Libraries

we use:
- numpy and pandas for data handling
- tensorflow/keras for building and training the model
- sklearn for evaluation metrics
- pickle to reload the saved tokenizer

In [1]:
import numpy as np
import pandas as pd
import pickle
import os

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, GlobalMaxPooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

In [3]:
DRIVE_DATA  = "/content/drive/MyDrive/jigsaw-data/"

MAX_VOCAB   = 20000
MAX_LEN     = 200
EMBED_DIM   = 100    # GloVe dimension
LSTM_UNITS  = 64
DROPOUT     = 0.3
BATCH_SIZE  = 256
EPOCHS      = 10

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## Load Pre-processed Data

Load the tokenized sequences and labels saved by `tokenization.ipynb`.

In [4]:
X_train = np.load(DRIVE_DATA + 'X_train.npy')
X_test  = np.load(DRIVE_DATA + 'X_test.npy')
y_train = np.load(DRIVE_DATA + 'y_train.npy')

with open(DRIVE_DATA + 'tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")

X_train : (159571, 200)
X_test  : (153164, 200)
y_train : (159571, 6)


## Train / Validation Split

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Train size: {X_tr.shape[0]}")
print(f"Val size  : {X_val.shape[0]}")

Train size: 127656
Val size  : 31915


## GloVe Embedding Matrix

Load pre-trained GloVe 100d vectors and build an embedding matrix
aligned with the tokenizer vocabulary.
GloVe is downloaded once and cached in Google Drive for future sessions.

In [6]:
GLOVE_DIR  = DRIVE_DATA + 'glove/'
GLOVE_PATH = GLOVE_DIR + 'glove.6B.100d.txt'

if not os.path.exists(GLOVE_PATH):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    !wget -q --show-progress -P {GLOVE_DIR} http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q {GLOVE_DIR}glove.6B.zip -d {GLOVE_DIR}
    !rm {GLOVE_DIR}glove.6B.zip
    print('GloVe downloaded and cached to Drive.')
else:
    print('GloVe found in Drive, skipping download.')

glove_index = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_index[word] = vector

print(f"Loaded {len(glove_index)} GloVe vectors")

glove.6B.zip        100%[===================>] 822.24M  4.46MB/s    in 2m 39s  
GloVe downloaded and cached to Drive.
Loaded 400000 GloVe vectors


In [7]:
embedding_matrix = np.zeros((MAX_VOCAB, EMBED_DIM))
covered = 0

for word, idx in tokenizer.word_index.items():
    if idx >= MAX_VOCAB:
        continue
    vector = glove_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
        covered += 1

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Words covered by GloVe: {covered} / {MAX_VOCAB}")

Embedding matrix shape: (20000, 100)
Words covered by GloVe: 18577 / 20000


## BiLSTM Architecture

Model design:
- Pre-trained GloVe embedding (frozen)
- Bidirectional LSTM to capture context from both directions
- GlobalMaxPooling to aggregate sequence features
- Dense output layer with sigmoid activation for multi-label classification

In [8]:
inp = Input(shape=(MAX_LEN,))

x = Embedding(
    input_dim=MAX_VOCAB,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix],
    trainable=False
)(inp)

x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
x = GlobalMaxPooling1D()(x)
x = Dropout(DROPOUT)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(DROPOUT)(x)
out = Dense(6, activation='sigmoid')(x)

model = Model(inputs=inp, outputs=out, name='bilstm_toxic')
model.summary()

Model: "bilstm_toxic"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 100)       │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 128)       │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,093,126 (7.98 MB)

 Trainable params: 93,126 (363.77 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

## Compile Model

In [9]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Callbacks

Best model checkpoint is saved to Google Drive so it persists across sessions.

In [10]:
MODEL_PATH = DRIVE_DATA + 'bilstm_best.keras'

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor='val_loss',
    save_best_only=True
)

print(f"Model will be saved to: {MODEL_PATH}")

Model will be saved to: /content/drive/MyDrive/jigsaw-data/bilstm_best.keras


## Train Model

In [11]:
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 26s 36ms/step - accuracy: 0.7747 - loss: 0.0954 - val_accuracy: 0.9941 - val_loss: 0.0574
Epoch 2/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9494 - loss: 0.0596 - val_accuracy: 0.9941 - val_loss: 0.0521
Epoch 3/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 18s 37ms/step - accuracy: 0.9678 - loss: 0.0548 - val_accuracy: 0.9941 - val_loss: 0.0494
Epoch 4/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.9746 - loss: 0.0521 - val_accuracy: 0.9941 - val_loss: 0.0481
Epoch 5/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 18s 37ms/step - accuracy: 0.9786 - loss: 0.0498 - val_accuracy: 0.9940 - val_loss: 0.0486
Epoch 6/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.9788 - loss: 0.0483 - val_accuracy: 0.9939 - val_loss: 0.0468
Epoch 7/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9750 - loss: 0.0471 - val_accuracy: 0.9941 - val_loss: 0.0464
Epoch 8/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9751 - loss: 0.0458 - 

## Evaluation — ROC-AUC

Compute per-label and mean ROC-AUC on the validation set.

In [ ]:
y_pred_val = model.predict(X_val, batch_size=BATCH_SIZE)

auc_scores = {}
for i, label in enumerate(labels):
    auc = roc_auc_score(y_val[:, i], y_pred_val[:, i])
    auc_scores[label] = auc
    print(f"{label:<15} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(list(auc_scores.values())):.4f}")

## Generate Test Predictions

In [ ]:
y_pred_test = model.predict(X_test, batch_size=BATCH_SIZE)
print(f"Predictions shape: {y_pred_test.shape}")

## Save Submission File to Google Drive

In [14]:
test_df = pd.read_csv("/content/drive/MyDrive/jigsaw-data/test.csv/test.csv")

submission = pd.DataFrame(y_pred_test, columns=labels)
submission.insert(0, 'id', test_df['id'])

SUBMISSION_PATH = DRIVE_DATA + 'submission_bilstm.csv'
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")
submission.head(2)

Submission saved to: /content/drive/MyDrive/jigsaw-data/submission_bilstm.csv


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.992914,3.307193e-01,0.924253,1.140407e-01,0.827874,0.391927
1,0000247867823ef7,0.001116,1.587180e-07,0.000142,1.140411e-07,0.000073,0.000006


## Key Findings

- BiLSTM with frozen GloVe embeddings achieved a mean validation ROC-AUC of ~0.978.
- Early stopping triggered at epoch 4, preventing overfitting.
- `threat` is the hardest label to classify, consistent with its low prevalence.
- `obscene` and `severe_toxic` are the easiest, likely due to distinctive vocabulary.
- Best model and submission saved to Google Drive.

### Next Steps

- Fine-tune embeddings (set `trainable=True`)
- Experiment with stacked BiLSTM layers
- Try attention mechanism over LSTM outputs
- Explore transformer-based models (BERT)